# Temporary Jacobi Row/Kernel Filtering Plots

Generated: `20260630_010028`

This notebook is intentionally timestamped and fixed to explicit RUN_IDs. It does not depend on `latest`.


In [ ]:
from pathlib import Path
import math
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TIMESTAMP = "20260630_010028"
DATA_ROOT = Path("/astrum/home/hpchzy/code/data")
HOST = "cgnr6760pn2"
NP = 64
NSAMP_RATIO = 0.5
REF_TIMER = "tsc"
TIMERS = ["cgt", "papi", "papix6", "wtime", "tsc", "tsc_native"]
SHUFFLES = ["shuffle0", "shuffle1", "shuffle2"]
PLOT_DIR = Path("stencil/plots_tmp_jacobi_row_kernel_" + TIMESTAMP)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ROW_KERNEL_1024 = DATA_ROOT / "20260629/cgnr6760pn2/output_jacobi_kernel_timing_0629_dsub/20260629-jacobi-row-vs-kernel-dsub"
RUN_KERNEL_32 = DATA_ROOT / "20260630/cgnr6760pn2/output_jacobi_kernel_size32_0630_dsub/20260630-jacobi-kernel-size32-dsub"
RUN_KERNEL_64_256 = DATA_ROOT / "20260630/cgnr6760pn2/output_jacobi_kernel_sizes64_256_0630_dsub/20260630-jacobi-kernel-sizes64-256-dsub"

RUN_SPECS = [
    dict(label="row1024", run=RUN_ROW_KERNEL_1024, variant="row_after", granularity="row", prefix="jacobi2d5p", sizes=[1024]),
    dict(label="kernel1024", run=RUN_ROW_KERNEL_1024, variant="kernel_after", granularity="kernel", prefix="jacobi2d5p_kernel", sizes=[1024]),
    dict(label="kernel32", run=RUN_KERNEL_32, variant="kernel_after", granularity="kernel", prefix="jacobi2d5p_kernel", sizes=[32]),
    dict(label="kernel64_256", run=RUN_KERNEL_64_256, variant="kernel_after", granularity="kernel", prefix="jacobi2d5p_kernel", sizes=[64,128,256]),
]

print("notebook_timestamp=", TIMESTAMP)
print("plot_dir=", PLOT_DIR)
for spec in RUN_SPECS:
    print(spec["label"], spec["run"], spec["run"].exists())


In [ ]:
def read_time_csv_dir(path, col):
    path = Path(path)
    vals = []
    files = sorted(path.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"no CSV files in {path}")
    for f in files:
        arr = np.loadtxt(f, delimiter=",")
        if arr.ndim == 1:
            if arr.size > col:
                vals.append(np.array([arr[col]], dtype=float))
        elif arr.shape[1] > col:
            vals.append(arr[:, col].astype(float))
    if not vals:
        raise RuntimeError(f"no values at column {col} in {path}")
    return np.concatenate(vals)


def read_hist(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    tcol = cols.get("t") or cols.get("time") or df.columns[0]
    pcol = cols.get("p") or df.columns[-1]
    t = df[tcol].to_numpy(float)
    p = df[pcol].to_numpy(float)
    if p.sum() > 0:
        p = p / p.sum()
    return t, p


def weighted_median_hist(path):
    t, p = read_hist(path)
    order = np.argsort(t)
    t = t[order]
    p = p[order]
    c = np.cumsum(p)
    return float(t[np.searchsorted(c, 0.5)])


def w1_hist(path_a, path_b):
    ta, pa = read_hist(path_a)
    tb, pb = read_hist(path_b)
    grid = np.unique(np.concatenate([ta, tb]))
    if len(grid) <= 1:
        return 0.0
    def cdf_on(t, p):
        order = np.argsort(t)
        t = t[order]
        p = p[order]
        c = np.cumsum(p)
        idx = np.searchsorted(t, grid, side="right") - 1
        out = np.zeros_like(grid, dtype=float)
        m = idx >= 0
        out[m] = c[idx[m]]
        return out
    ca = cdf_on(ta, pa)
    cb = cdf_on(tb, pb)
    return float(np.sum(np.abs(ca[:-1] - cb[:-1]) * np.diff(grid)))


def plot_hist_cdf(ax, hist_path, label, **kwargs):
    t, p = read_hist(hist_path)
    order = np.argsort(t)
    t = t[order]
    p = p[order]
    c = np.cumsum(p)
    ax.plot(t, c, label=label, **kwargs)


def read_float(path):
    path = Path(path)
    return float(path.read_text().strip()) if path.exists() else np.nan


def find_one(pattern, base):
    matches = sorted(Path(base).glob(pattern))
    if not matches:
        raise FileNotFoundError(f"missing pattern {pattern} under {base}")
    if len(matches) > 1:
        print(f"warning: multiple matches for {pattern}; using {matches[0]}")
    return matches[0]


In [ ]:
rows = []
missing = []
for spec in RUN_SPECS:
    if not spec["run"].exists():
        raise FileNotFoundError(spec["run"])
    for size in spec["sizes"]:
        for shuffle in SHUFFLES:
            shuffle_dir = spec["run"] / spec["variant"] / shuffle
            if not shuffle_dir.exists():
                missing.append(str(shuffle_dir))
                continue
            for timer in TIMERS:
                tm_dir = shuffle_dir / f'{spec["prefix"]}_{timer}_np{NP}_size{size}'
                if not tm_dir.exists():
                    missing.append(str(tm_dir))
                    continue
                tf_dir = find_one(f'{spec["prefix"]}_{timer}_np{NP}_size{size}_nsampRatio{NSAMP_RATIO}_nsamp*_tf', shuffle_dir)
                filt_dir = find_one(f'{spec["prefix"]}_{timer}_np{NP}_size{size}_nsampRatio{NSAMP_RATIO}_nsamp*_filt', shuffle_dir)
                tm_seq = read_time_csv_dir(tm_dir, 1)
                te_seq = read_time_csv_dir(tf_dir, 2)
                rows.append(dict(
                    run_label=spec["label"],
                    run_id=spec["run"].name,
                    variant=spec["variant"],
                    granularity=spec["granularity"],
                    size=size,
                    shuffle=shuffle,
                    timer=timer,
                    tm_n=len(tm_seq),
                    te_n=len(te_seq),
                    tm_median_ns=float(np.median(tm_seq)),
                    te_median_ns=float(np.median(te_seq)),
                    tm_p95_ns=float(np.percentile(tm_seq, 95)),
                    te_p95_ns=float(np.percentile(te_seq, 95)),
                    tr_median_ns=weighted_median_hist(filt_dir / "tr_hist.csv"),
                    tm_hist=str(filt_dir / "tm_hist.csv"),
                    tr_hist=str(filt_dir / "tr_hist.csv"),
                    sim_cdf=str(filt_dir / "sim_cdf.csv"),
                    er=read_float(filt_dir / "er.out"),
                    ep=read_float(filt_dir / "ep.out"),
                ))

if missing:
    raise RuntimeError("missing required data:\n" + "\n".join(missing[:20]))

df = pd.DataFrame(rows)
print("loaded rows", len(df))
print(df.groupby(["granularity","size","timer"]).size().unstack(fill_value=0))
df.to_csv(PLOT_DIR / "jacobi_row_kernel_metrics_raw.csv", index=False)


In [ ]:
# Add divergence to TSC and aggregate summaries.
div_rows = []
for (granularity, size, shuffle), g in df.groupby(["granularity", "size", "shuffle"]):
    ref = g[g.timer == REF_TIMER]
    if ref.empty:
        raise RuntimeError(f"missing {REF_TIMER} for {granularity} size={size} {shuffle}")
    ref = ref.iloc[0]
    for _, row in g.iterrows():
        div_rows.append(dict(
            granularity=granularity,
            size=size,
            shuffle=shuffle,
            timer=row.timer,
            tm_w1_to_tsc_ns=w1_hist(row.tm_hist, ref.tm_hist),
            tr_w1_to_tsc_ns=w1_hist(row.tr_hist, ref.tr_hist),
            tm_ratio_to_tsc=row.tm_median_ns / ref.tm_median_ns,
            te_ratio_to_tsc=row.te_median_ns / ref.te_median_ns,
            tr_ratio_to_tsc=row.tr_median_ns / ref.tr_median_ns,
        ))
div = pd.DataFrame(div_rows)
df2 = df.merge(div, on=["granularity","size","shuffle","timer"], how="left")

summary_rows = []
for (granularity, size, timer), g in df2.groupby(["granularity","size","timer"], sort=False):
    d = dict(granularity=granularity, size=int(size), timer=timer)
    for col in ["tm_median_ns","te_median_ns","tr_median_ns","tm_w1_to_tsc_ns","tr_w1_to_tsc_ns","tm_ratio_to_tsc","te_ratio_to_tsc","tr_ratio_to_tsc","er","ep"]:
        d[col] = float(g[col].median())
    for col in ["tm_median_ns","te_median_ns","tr_median_ns"]:
        vals = g[col].to_numpy(float)
        prefix = col.replace("_median_ns", "")
        d[f"{prefix}_shuffle_cv"] = float(np.std(vals, ddof=1) / np.mean(vals))
        d[f"{prefix}_shuffle_range_ns"] = float(np.max(vals) - np.min(vals))
    summary_rows.append(d)
summary = pd.DataFrame(summary_rows)

spread_rows = []
for (granularity, size), g in summary.groupby(["granularity","size"]):
    nonref = g[g.timer != REF_TIMER]
    tm_sum = nonref.tm_w1_to_tsc_ns.sum()
    tr_sum = nonref.tr_w1_to_tsc_ns.sum()
    spread_rows.append(dict(
        granularity=granularity,
        size=int(size),
        tm_timer_spread_ratio=float(g.tm_median_ns.max()/g.tm_median_ns.min() - 1),
        te_timer_spread_ratio=float(g.te_median_ns.max()/g.te_median_ns.min() - 1),
        tr_timer_spread_ratio=float(g.tr_median_ns.max()/g.tr_median_ns.min() - 1),
        median_tm_shuffle_cv=float(g.tm_shuffle_cv.median()),
        median_te_shuffle_cv=float(g.te_shuffle_cv.median()),
        median_tr_shuffle_cv=float(g.tr_shuffle_cv.median()),
        median_tm_w1_to_tsc_ns=float(g.tm_w1_to_tsc_ns.median()),
        median_tr_w1_to_tsc_ns=float(g.tr_w1_to_tsc_ns.median()),
        median_tm_w1_rel=float((g.tm_w1_to_tsc_ns / g[g.timer == REF_TIMER].tm_median_ns.iloc[0]).median()),
        median_tr_w1_rel=float((g.tr_w1_to_tsc_ns / g[g.timer == REF_TIMER].tr_median_ns.iloc[0]).median()),
        w1_reduction_sum=float(1 - tr_sum / tm_sum) if tm_sum else np.nan,
        max_ep=float(g.ep.max()),
        max_er=float(g.er.max()),
    ))
spread = pd.DataFrame(spread_rows).sort_values(["granularity","size"])

df2.to_csv(PLOT_DIR / "jacobi_row_kernel_metrics.csv", index=False)
summary.to_csv(PLOT_DIR / "jacobi_row_kernel_summary.csv", index=False)
spread.to_csv(PLOT_DIR / "jacobi_row_kernel_spread_reduction.csv", index=False)

print("summary")
print(summary[["granularity","size","timer","tm_median_ns","te_median_ns","tr_median_ns","tm_ratio_to_tsc","tr_ratio_to_tsc","er","ep"]].sort_values(["granularity","size","timer"]).to_string(index=False))
print("\nspread")
print(spread.to_string(index=False))


In [ ]:
# Row vs kernel, size=1024: TM/TE/TR median by timer.
plt.style.use("seaborn-v0_8-whitegrid")
sub = summary[summary["size"] == 1024].copy()
timer_order = TIMERS
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, title in zip(axes, ["tm_median_ns","te_median_ns","tr_median_ns"], ["TM median", "TE median", "TR median"]):
    pivot = sub.pivot(index="timer", columns="granularity", values=metric).reindex(timer_order)
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"Jacobi size=1024: {title}")
    ax.set_xlabel("timer")
    ax.set_ylabel("ns")
    ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
out = PLOT_DIR / "jacobi_row_vs_kernel_size1024_tm_te_tr_median.png"
fig.savefig(out, dpi=180)
print(out)


In [ ]:
# Row vs kernel, size=1024: W1 divergence before/after filtering.
sub = summary[(summary["size"] == 1024) & (summary.timer != REF_TIMER)].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)
for ax, granularity in zip(axes, ["row", "kernel"]):
    g = sub[sub.granularity == granularity].set_index("timer").reindex([t for t in TIMERS if t != REF_TIMER])
    g[["tm_w1_to_tsc_ns", "tr_w1_to_tsc_ns"]].plot(kind="bar", ax=ax)
    ax.set_title(f"{granularity}: W1 to {REF_TIMER}, size=1024")
    ax.set_xlabel("timer")
    ax.set_ylabel("W1 distance (ns)")
    ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
out = PLOT_DIR / "jacobi_row_vs_kernel_size1024_w1_divergence.png"
fig.savefig(out, dpi=180)
print(out)

red = spread[spread["size"] == 1024].copy()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(red.granularity, red.w1_reduction_sum)
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("1 - sum(W1_TR) / sum(W1_TM)")
ax.set_title("Filtering divergence reduction, size=1024")
fig.tight_layout()
out = PLOT_DIR / "jacobi_row_vs_kernel_size1024_w1_reduction.png"
fig.savefig(out, dpi=180)
print(out)


In [ ]:
# Row vs kernel, size=1024: EP/ER and shuffle CV.
sub = summary[summary["size"] == 1024].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric, title in zip(axes, ["ep", "er"], ["EP", "ER"]):
    pivot = sub.pivot(index="timer", columns="granularity", values=metric).reindex(TIMERS)
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"{title} by timer, size=1024")
    ax.set_xlabel("timer")
    ax.set_ylabel(title)
    ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
out = PLOT_DIR / "jacobi_row_vs_kernel_size1024_ep_er.png"
fig.savefig(out, dpi=180)
print(out)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, title in zip(axes, ["tm_shuffle_cv","te_shuffle_cv","tr_shuffle_cv"], ["TM shuffle CV", "TE shuffle CV", "TR shuffle CV"]):
    pivot = sub.pivot(index="timer", columns="granularity", values=metric).reindex(TIMERS)
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"{title}, size=1024")
    ax.set_xlabel("timer")
    ax.set_ylabel("CV")
    ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
out = PLOT_DIR / "jacobi_row_vs_kernel_size1024_shuffle_cv.png"
fig.savefig(out, dpi=180)
print(out)


In [ ]:
# CDF overlays for row vs kernel at size=1024, one plot per timer.
for timer in TIMERS:
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    for granularity, color in [("row", "tab:blue"), ("kernel", "tab:orange")]:
        sub = df2[(df2["size"] == 1024) & (df2.timer == timer) & (df2.granularity == granularity) & (df2.shuffle == "shuffle0")]
        if sub.empty:
            continue
        r = sub.iloc[0]
        plot_hist_cdf(ax, r.tm_hist, f"{granularity} TM", color=color, linestyle="-")
        plot_hist_cdf(ax, r.tr_hist, f"{granularity} TR", color=color, linestyle="--")
    ax.set_title(f"Jacobi size=1024 CDF: {timer}")
    ax.set_xlabel("time (ns)")
    ax.set_ylabel("CDF")
    ax.legend()
    fig.tight_layout()
    out = PLOT_DIR / f"jacobi_row_vs_kernel_size1024_cdf_{timer}.png"
    fig.savefig(out, dpi=180)
    print(out)


In [ ]:
# Kernel-level size sweep: 32, 64, 128, 256, 1024.
ks = summary[summary.granularity == "kernel"].copy()
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
for ax, metric, title in zip(axes, ["tm_median_ns","te_median_ns","tr_median_ns"], ["TM median", "TE median", "TR median"]):
    pivot = ks.pivot(index="size", columns="timer", values=metric).sort_index()[TIMERS]
    pivot.plot(marker="o", ax=ax)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_title(f"Kernel-level Jacobi: {title}")
    ax.set_xlabel("size")
    ax.set_ylabel("ns")
fig.tight_layout()
out = PLOT_DIR / "jacobi_kernel_size_sweep_tm_te_tr_by_timer.png"
fig.savefig(out, dpi=180)
print(out)

sp = spread[spread.granularity == "kernel"].sort_values("size")
fig, ax = plt.subplots(figsize=(8, 5))
for metric, label in [("tm_timer_spread_ratio", "TM timer spread"), ("te_timer_spread_ratio", "TE timer spread"), ("tr_timer_spread_ratio", "TR timer spread")]:
    ax.plot(sp["size"], sp[metric], marker="o", label=label)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("size")
ax.set_ylabel("max/min - 1")
ax.set_title("Kernel-level Jacobi timer spread vs size")
ax.legend()
fig.tight_layout()
out = PLOT_DIR / "jacobi_kernel_size_sweep_timer_spread.png"
fig.savefig(out, dpi=180)
print(out)


In [ ]:
# Kernel-level size sweep: relative W1 and filtering reduction.
sp = spread[spread.granularity == "kernel"].sort_values("size")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sp["size"], sp["median_tm_w1_rel"], marker="o", label="TM median W1 / TSC median")
axes[0].plot(sp["size"], sp["median_tr_w1_rel"], marker="o", label="TR median W1 / TSC median")
axes[0].set_xscale("log", base=2)
axes[0].set_yscale("log")
axes[0].set_xlabel("size")
axes[0].set_ylabel("relative W1")
axes[0].set_title("Kernel-level relative divergence vs size")
axes[0].legend()

axes[1].plot(sp["size"], sp["w1_reduction_sum"], marker="o")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("size")
axes[1].set_ylabel("1 - sum(W1_TR)/sum(W1_TM)")
axes[1].set_title("Filtering W1 reduction vs size")
fig.tight_layout()
out = PLOT_DIR / "jacobi_kernel_size_sweep_w1_reduction.png"
fig.savefig(out, dpi=180)
print(out)

print("Generated CSVs:")
for p in sorted(PLOT_DIR.glob("*.csv")):
    print(p)
print("Generated PNGs:")
for p in sorted(PLOT_DIR.glob("*.png")):
    print(p)


In [ ]:

# 0617-like core figures: each size one figure; all timers with TM and TR side-by-side.
TIMER_ORDER = ["cgt", "papi", "papix6", "wtime", "tsc", "tsc_native"]
for size in sorted(summary["size"].unique()):
    sub = summary[summary["size"] == size].copy()
    grans = list(sub["granularity"].drop_duplicates())
    grans = sorted(grans, key=lambda x: 0 if x == "row" else 1)
    fig, axes = plt.subplots(1, len(grans), figsize=(8 * len(grans), 5), sharey=False)
    if len(grans) == 1:
        axes = [axes]
    for ax, gran in zip(axes, grans):
        g = sub[sub["granularity"] == gran].set_index("timer").reindex(TIMER_ORDER)
        plot_df = g[["tm_median_ns", "tr_median_ns"]].rename(columns={"tm_median_ns": "TM", "tr_median_ns": "TR"})
        plot_df.plot(kind="bar", ax=ax, width=0.78)
        ax.set_title(f"Jacobi {gran}-level size={size}: TM vs TR")
        ax.set_xlabel("timer")
        ax.set_ylabel("time (ns)")
        ax.tick_params(axis="x", rotation=35)
        ax.grid(axis="y", alpha=0.35)
    fig.tight_layout()
    out = PLOT_DIR / f"jacobi_tm_tr_by_timer_size{int(size)}.png"
    fig.savefig(out, dpi=220)
    print(out)

# Normalized companion: each timer relative to TSC for that granularity and size.
for size in sorted(summary["size"].unique()):
    sub = summary[summary["size"] == size].copy()
    grans = list(sub["granularity"].drop_duplicates())
    grans = sorted(grans, key=lambda x: 0 if x == "row" else 1)
    fig, axes = plt.subplots(1, len(grans), figsize=(8 * len(grans), 5), sharey=True)
    if len(grans) == 1:
        axes = [axes]
    for ax, gran in zip(axes, grans):
        g = sub[sub["granularity"] == gran].set_index("timer").reindex(TIMER_ORDER)
        ref_tm = float(g.loc["tsc", "tm_median_ns"])
        ref_tr = float(g.loc["tsc", "tr_median_ns"])
        plot_df = pd.DataFrame({"TM/TSC_TM": g["tm_median_ns"] / ref_tm, "TR/TSC_TR": g["tr_median_ns"] / ref_tr})
        plot_df.plot(kind="bar", ax=ax, width=0.78)
        ax.axhline(1.0, color="black", lw=1)
        ax.set_title(f"Jacobi {gran}-level size={size}: normalized TM/TR")
        ax.set_xlabel("timer")
        ax.set_ylabel("ratio to TSC")
        ax.tick_params(axis="x", rotation=35)
        ax.grid(axis="y", alpha=0.35)
    fig.tight_layout()
    out = PLOT_DIR / f"jacobi_tm_tr_by_timer_size{int(size)}_normalized.png"
    fig.savefig(out, dpi=220)
    print(out)


In [ ]:

# Correct 0617-style FILT CDF figures:
# one size per figure; each figure contains all timers, with TM as dashed-dot and TR as solid-triangle.
TIMER_ORDER = ["cgt", "papi", "papix6", "wtime", "tsc", "tsc_native"]
TIMER_COLORS_0617 = {
    "cgt": "tab:blue",
    "papi": "tab:orange",
    "papix6": "tab:green",
    "wtime": "tab:red",
    "tsc": "tab:purple",
    "tsc_native": "tab:brown",
}
TIMER_LABELS_0617 = {
    "cgt": "CGT",
    "papi": "PAPI",
    "papix6": "PAPIx6",
    "wtime": "MPI_Wtime",
    "tsc": "TSC",
    "tsc_native": "TSC-native",
}
QUANTILE_DROP_0617 = 0.995

def _hist_cdf_0617(path):
    t, p = read_hist(path)
    order = np.argsort(t)
    t = t[order]
    p = p[order]
    c = np.cumsum(p)
    keep = c <= QUANTILE_DROP_0617
    if len(c) and (not keep.any() or keep.sum() < len(c)):
        first_over = np.searchsorted(c, QUANTILE_DROP_0617)
        keep[: min(first_over + 1, len(keep))] = True
    return t[keep], c[keep]

def _ordered_tr_tm_0617(ax):
    h, l = ax.get_legend_handles_labels()
    tr = [(hh, ll) for hh, ll in zip(h, l) if ll.endswith("-TR")]
    tm = [(hh, ll) for hh, ll in zip(h, l) if ll.endswith("-TM")]
    ordered = tr + tm
    return [hh for hh, _ in ordered], [ll for _, ll in ordered]

def _plot_0617_group(ax, group, title):
    for timer in TIMER_ORDER:
        rows = group[group["timer"] == timer]
        if rows.empty:
            continue
        row = rows.iloc[0]
        color = TIMER_COLORS_0617[timer]
        label = TIMER_LABELS_0617[timer]
        tm_x, tm_y = _hist_cdf_0617(row["tm_hist"])
        tr_x, tr_y = _hist_cdf_0617(row["tr_hist"])
        ax.plot(tm_x, tm_y, label=f"{label}-TM", color=color, linestyle="--",
                linewidth=1.0, marker=".", markevery=max(1, len(tm_x)//50),
                markersize=10, markerfacecolor="none", markeredgewidth=0.8)
        ax.plot(tr_x, tr_y, label=f"{label}-TR", color=color, linestyle="-",
                linewidth=1.0, marker="^", markevery=max(1, len(tr_x)//50),
                markersize=7, markerfacecolor="none", markeredgewidth=0.8)
    ax.set_title(title, fontsize=18)
    ax.set_xlabel("Time (ns)", fontsize=14)
    ax.set_ylabel("CDF", fontsize=14)
    ax.set_ylim(-0.03, QUANTILE_DROP_0617 + 0.03)
    ax.grid(True, alpha=0.28)
    ax.tick_params(axis="both", labelsize=12)
    oh, ol = _ordered_tr_tm_0617(ax)
    if oh:
        ax.legend(oh, ol, ncol=2, fontsize=9, loc="lower right")

for granularity in ["kernel", "row"]:
    gdf = df2[df2["granularity"] == granularity]
    for size in sorted(gdf["size"].unique()):
        group = gdf[(gdf["size"] == size) & (gdf["shuffle"] == "shuffle0")]
        if group.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
        _plot_0617_group(ax, group, f"Jacobi {granularity}-level size={int(size)}: TM/TR CDF by timer")
        out = PLOT_DIR / f"jacobi_0617style_{granularity}_size{int(size)}_tm_tr_cdf.png"
        fig.savefig(out, dpi=220, bbox_inches="tight")
        print(out)

sub = df2[(df2["size"] == 1024) & (df2["shuffle"] == "shuffle0")]
fig, axes = plt.subplots(1, 2, figsize=(20, 7), sharey=True, constrained_layout=True)
for ax, granularity in zip(axes, ["row", "kernel"]):
    _plot_0617_group(ax, sub[sub["granularity"] == granularity],
                     f"Jacobi {granularity}-level size=1024: TM/TR CDF by timer")
out = PLOT_DIR / "jacobi_0617style_row_kernel_size1024_tm_tr_cdf.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
print(out)
